# RQ1/RQ2/RQ3 GPU run on Google Colab (one pinned environment)
**Runtime → Change runtime type → T4 GPU.** Run the cells from top to bottom.
Results are copied to your Google Drive after every stage (and every 10 minutes), so if Colab disconnects, open this notebook again, run cells 1-4 and then cell 5 again: finished work is skipped.
Expected time on a T4: about 9-10 hours in total, spread over several sessions.

In [ ]:
# 1. GPU check
!nvidia-smi
import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then run again'

In [ ]:
# 2. Google Drive. Put EITHER the file RQ1_FINAL.zip OR the folder RQ1_FINAL into My Drive/RQ1_GPU/ first.
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ZIP = '/content/drive/MyDrive/RQ1_GPU/RQ1_FINAL.zip'
DRIVE_CODE = '/content/drive/MyDrive/RQ1_GPU/RQ1_FINAL'
DRIVE_RESULTS = '/content/drive/MyDrive/RQ1_GPU/results_pinned'
shutil.rmtree('/content/RQ1_FINAL', ignore_errors=True)
if os.path.exists(DRIVE_ZIP):
    shutil.unpack_archive(DRIVE_ZIP, '/content')
elif os.path.isdir(DRIVE_CODE):
    shutil.copytree(DRIVE_CODE, '/content/RQ1_FINAL', ignore=shutil.ignore_patterns('results', '__pycache__'))
else:
    raise SystemExit('Neither ' + DRIVE_ZIP + ' nor the folder ' + DRIVE_CODE + ' exists')
assert os.path.isdir('/content/RQ1_FINAL/data/eval_v7/frozen_seeds'), 'RQ1_FINAL/data/eval_v7/frozen_seeds missing (the zip must contain the folder RQ1_FINAL at its top level)'
assert os.path.exists('/content/RQ1_FINAL/data/PII_SCRUBBED'), 'This is not the scrubbed data copy (data/PII_SCRUBBED is missing)'
os.chdir('/content/RQ1_FINAL'); os.makedirs('results', exist_ok=True)
print(os.getcwd(), len(os.listdir('.')), 'files')

In [ ]:
# 3. Install the pinned versions (Colab's own torch/transformers are kept)
!pip install -q -r requirements_gpu_pinned.txt
import importlib, sentence_transformers, transformers
print('sentence-transformers', sentence_transformers.__version__, '| transformers', transformers.__version__, '| torch', torch.__version__)
assert sentence_transformers.__version__ == '5.7.0', 'Restart the runtime (Runtime > Restart session), then run cells 1-4 again'

In [ ]:
# 4. Quick self-test (about 5 minutes): environment report + embeddings of all encoders. Must end with: FAILED STAGES: none
%run run_everything_gpu.py --only env,embeddings --sync "$DRIVE_RESULTS"

In [ ]:
# 5. THE MAIN RUN. All remaining stages in priority order. If the session stops, run cells 1-4 and then this cell again.
%run run_everything_gpu.py --skip env,embeddings --sync "$DRIVE_RESULTS"

In [ ]:
# 6. At the end: what is finished and what failed. (Re-run cell 5 until it prints: FAILED STAGES: none)
!ls "$DRIVE_RESULTS"
!cat results/ENVIRONMENT.txt | head -40

In [ ]:
# 7. Pack the results for me (one file on your Drive; upload it to the chat)
!cd /content/RQ1_FINAL && zip -qr "$DRIVE_RESULTS/../results_for_paper.zip" results -x "results/embeddings/*" -x "results/colab_v7b/*" -x "results/colab_v7b_chunked/*"
!ls -lh "$DRIVE_RESULTS/../results_for_paper.zip"